# Skin Cancer — Notebook 2 : évaluation
Suite du notebook d'entraînement `skincanerf` (aucun entraînement ici).

**Partie A** : sans GPU — PH2 calibré + intervalles de confiance (quelques secondes).
**Partie B** : GPU requis — ensemble des checkpoints. Les cellules de la partie B **se sautent automatiquement** s'il n'y a pas de GPU, donc « Run All » ne peut plus bloquer 12 h.

**Input requis** : l'Output du notebook `skincanerf` (+ les 3 datasets d'images seulement pour la partie B).

## Cellule 1 — Setup

In [1]:
import os, random, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, mixed_precision
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

GPUS    = tf.config.list_physical_devices("GPU")
HAS_GPU = len(GPUS) > 0
if HAS_GPU:
    mixed_precision.set_global_policy("mixed_float16")   # float16 seulement sur GPU (très lent sur CPU)

IMG_SIZE, BATCH_SIZE, NUM_CLASSES = 384, 32, 7
EVAL_RESIZE = int(IMG_SIZE * 1.15)
AUTOTUNE    = tf.data.AUTOTUNE
PREPROCESS  = lambda x: x
CLASSES     = ["akiec", "bcc", "bkl", "df", "nv", "mel", "vasc"]
NV, MEL     = 4, 5

INPUT_DIR = "/kaggle/input/notebooks/rihembousbih/skincanerf"
assert os.path.exists(INPUT_DIR), (
    f"❌ {INPUT_DIR} introuvable → panneau de droite : + Add Input → onglet Notebooks → 'skincanerf'")

print("GPU :", "✅ disponible" if HAS_GPU else "❌ aucun (partie A seulement, la partie B sera sautée)")
print("Fichiers dans INPUT_DIR :", sorted(os.listdir(INPUT_DIR)))

GPU : ❌ aucun (partie A seulement, la partie B sera sautée)
Fichiers dans INPUT_DIR : ['__notebook__.ipynb', '__output__.json', '__results__.html', 'ckpt_p2a.zip', 'class_mapping.json', 'custom.css', 'ensemble_weights.npy', 'predictions_test.csv', 'probs_ph2.npy', 'probs_test.npy', 'probs_val.npy', 'resnet50_cbam_final.keras', 'resnet50_cbam_finetune_step1.keras', 'resnet50_cbam_finetune_step2.keras', 'resnet50_cbam_phase1.keras', 'resnet50_cbam_phase1_final.keras', 'run_config.json', 'split_ph2.csv', 'split_test.csv', 'split_train.csv', 'split_val.csv', 'temperature.npy', 'thresholds.npy']


2026-09-22 21:25:26.357317: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


# PARTIE A — sans GPU
## Cellule 2 — PH2 calibré + intervalles de confiance (bootstrap)

In [2]:
probs_val  = np.load(f"{INPUT_DIR}/probs_val.npy")
probs_test = np.load(f"{INPUT_DIR}/probs_test.npy")
probs_ph2  = np.load(f"{INPUT_DIR}/probs_ph2.npy")
T   = float(np.load(f"{INPUT_DIR}/temperature.npy")[0])
thr = np.load(f"{INPUT_DIR}/thresholds.npy")

y_test = pd.read_csv(f"{INPUT_DIR}/split_test.csv")["label"].values
y_ph2  = pd.read_csv(f"{INPUT_DIR}/split_ph2.csv")["label"].values

def apply_temperature(probs, T):
    logits = np.log(np.clip(probs, 1e-12, 1 - 1e-12))
    s = np.exp(logits / T)
    return s / s.sum(axis=1, keepdims=True)

def apply_thresholds(probs, thr):
    thr = np.clip(np.asarray(thr, dtype=float), 0.05, 5.0)
    return np.argmax(probs / thr[None, :], axis=1)

# ─── Métriques ───
acc    = lambda yt, yp: accuracy_score(yt, yp)
mf1    = lambda yt, yp: f1_score(yt, yp, labels=np.arange(7), average="macro", zero_division=0)
s_mel  = lambda yt, yp: ((yt == MEL) & (yp == MEL)).sum() / max((yt == MEL).sum(), 1)
sp_mel = lambda yt, yp: ((yt != MEL) & (yp != MEL)).sum() / max((yt != MEL).sum(), 1)

def bootstrap_ci(yt, yp, metric, n=2000, seed=42):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(yt), len(yt))
        vals.append(metric(yt[i], yp[i]))
    return np.percentile(vals, [2.5, 97.5])

def report(nom, yt, yp, metrics):
    print(f"\n=== {nom} ({len(yt)} images) ===")
    for mname, m in metrics.items():
        lo, hi = bootstrap_ci(yt, yp, m)
        print(f"  {mname:18s} {m(yt, yp):.4f}   IC95% [{lo:.4f} – {hi:.4f}]")

# ─── TEST : brut vs calibré ───
y_raw = probs_test.argmax(1)
y_cal = apply_thresholds(apply_temperature(probs_test, T), thr)
print(f"Contrôle : acc test brut = {acc(y_test, y_raw):.4f} (attendu 0.8285)")

M = {"Accuracy": acc, "Macro-F1": mf1, "Sensibilité mel": s_mel, "Spécificité mel": sp_mel}
report("TEST brut",    y_test, y_raw, M)
report("TEST calibré", y_test, y_cal, M)

# ─── PH2 : même point de fonctionnement que le TEST ───
y_ph2_raw = probs_ph2.argmax(1)
y_ph2_cal = apply_thresholds(apply_temperature(probs_ph2, T), thr)
yt_bin = (y_ph2 == MEL).astype(int)          # PH2 = nv/mel → tâche binaire mel vs non-mel
M_bin = {"Accuracy": acc,
         "Sensibilité mel": lambda a, b: ((a == 1) & (b == 1)).sum() / max((a == 1).sum(), 1),
         "Spécificité mel": lambda a, b: ((a == 0) & (b == 0)).sum() / max((a == 0).sum(), 1)}
report("PH2 brut (mel vs non-mel)",    yt_bin, (y_ph2_raw == MEL).astype(int), M_bin)
report("PH2 calibré (mel vs non-mel)", yt_bin, (y_ph2_cal == MEL).astype(int), M_bin)
print(f"\n  AUC mel PH2 (proba brute) : {roc_auc_score(yt_bin, probs_ph2[:, MEL]):.4f}")

Contrôle : acc test brut = 0.8285 (attendu 0.8285)

=== TEST brut (3738 images) ===
  Accuracy           0.8285   IC95% [0.8162 – 0.8406]
  Macro-F1           0.7349   IC95% [0.7022 – 0.7627]
  Sensibilité mel    0.7145   IC95% [0.6808 – 0.7479]
  Spécificité mel    0.9429   IC95% [0.9345 – 0.9507]

=== TEST calibré (3738 images) ===
  Accuracy           0.7865   IC95% [0.7745 – 0.7999]
  Macro-F1           0.7024   IC95% [0.6666 – 0.7319]
  Sensibilité mel    0.8362   IC95% [0.8084 – 0.8638]
  Spécificité mel    0.8425   IC95% [0.8296 – 0.8550]

=== PH2 brut (mel vs non-mel) (197 images) ===
  Accuracy           0.8680   IC95% [0.8223 – 0.9137]
  Sensibilité mel    0.5769   IC95% [0.4386 – 0.7115]
  Spécificité mel    0.9724   IC95% [0.9429 – 0.9934]

=== PH2 calibré (mel vs non-mel) (197 images) ===
  Accuracy           0.8680   IC95% [0.8173 – 0.9137]
  Sensibilité mel    0.7308   IC95% [0.6087 – 0.8478]
  Spécificité mel    0.9172   IC95% [0.8696 – 0.9605]

  AUC mel PH2 (proba bru

# PARTIE B — GPU requis (à lancer plus tard, avec le nouveau modèle final)
## Cellule 3 — Couches CBAM + fonctions image

In [3]:
class ChannelAttention(layers.Layer):
    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio
        self.gap = layers.GlobalAveragePooling2D()
        self.gmp = layers.GlobalMaxPooling2D()
        self.add = layers.Add()
        self.act = layers.Activation("sigmoid")
        self.mul = layers.Multiply()

    def build(self, input_shape):
        channels = int(input_shape[-1])
        hidden   = max(channels // self.ratio, 1)
        self.d1  = layers.Dense(hidden, activation="relu",
                                kernel_initializer="he_normal", use_bias=True, dtype="float32")
        self.d2  = layers.Dense(channels,
                                kernel_initializer="he_normal", use_bias=True, dtype="float32")
        self.d1.build((None, 1, 1, channels))
        self.d2.build((None, 1, 1, hidden))
        super().build(input_shape)

    def call(self, x):                       # version float32 (anti-NaN)
        in_dtype = x.dtype
        x32 = tf.cast(x, tf.float32)
        c   = int(x.shape[-1])
        avg = tf.reshape(tf.reduce_mean(x32, axis=[1, 2]), (-1, 1, 1, c))
        mx  = tf.reshape(tf.reduce_max(x32,  axis=[1, 2]), (-1, 1, 1, c))
        att = tf.sigmoid(self.d2(self.d1(avg)) + self.d2(self.d1(mx)))
        return tf.cast(x32 * att, in_dtype)

    def get_config(self):
        cfg = super().get_config(); cfg.update({"ratio": self.ratio}); return cfg


class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size
        self.mul = layers.Multiply()

    def build(self, input_shape):
        self.conv = layers.Conv2D(1, self.kernel_size, padding="same", activation="sigmoid",
                                  kernel_initializer="he_normal", use_bias=False, dtype="float32")
        self.conv.build((input_shape[0], input_shape[1], input_shape[2], 2))
        super().build(input_shape)

    def call(self, x):                       # version float32 (anti-NaN)
        in_dtype = x.dtype
        x32 = tf.cast(x, tf.float32)
        avg = tf.reduce_mean(x32, axis=-1, keepdims=True)
        mx  = tf.reduce_max(x32,  axis=-1, keepdims=True)
        att = self.conv(tf.concat([avg, mx], axis=-1))
        return tf.cast(x32 * att, in_dtype)

    def get_config(self):
        cfg = super().get_config(); cfg.update({"kernel_size": self.kernel_size}); return cfg


CUSTOM_OBJECTS = {"ChannelAttention": ChannelAttention, "SpatialAttention": SpatialAttention}


def decode_image(path):
    img_bytes = tf.io.read_file(path)
    ext = tf.strings.lower(tf.strings.split(path, ".")[-1])
    def _jpg(): return tf.image.decode_jpeg(img_bytes, channels=3)
    def _png(): return tf.image.decode_png(img_bytes,  channels=3)
    def _bmp(): return tf.image.decode_bmp(img_bytes)
    img = tf.case(
        [(tf.equal(ext, "jpg"),  _jpg), (tf.equal(ext, "jpeg"), _jpg),
         (tf.equal(ext, "png"),  _png), (tf.equal(ext, "bmp"),  _bmp)],
        default=_jpg, exclusive=True)
    return tf.ensure_shape(img, [None, None, 3])


def shades_of_gray(img, p=6.0):
    flat  = tf.reshape(img, [-1, 3])
    illum = tf.pow(tf.reduce_mean(tf.pow(flat + 1e-6, p), axis=0), 1.0 / p)
    illum = illum / (tf.norm(illum) + 1e-6)
    img   = img / (illum * tf.sqrt(3.0) + 1e-6)
    return tf.clip_by_value(img, 0.0, 255.0)


def predict_tta(model, df, n_rot=4):
    probs = np.zeros((len(df), NUM_CLASSES), dtype=np.float64)
    for k in range(n_rot):
        for flip in (False, True):
            def _map(p, y, k=k, flip=flip):
                img = tf.cast(decode_image(p), tf.float32)
                img = tf.image.resize(img, (EVAL_RESIZE, EVAL_RESIZE))
                off = (EVAL_RESIZE - IMG_SIZE) // 2
                img = tf.image.crop_to_bounding_box(img, off, off, IMG_SIZE, IMG_SIZE)
                img = tf.image.rot90(img, k=k)
                if flip:
                    img = tf.image.flip_left_right(img)
                img = shades_of_gray(img)
                return PREPROCESS(img), y
            ds = (tf.data.Dataset.from_tensor_slices(
                      (df["path"].values.astype(str), df["label"].values.astype(np.int32)))
                  .map(_map, num_parallel_calls=AUTOTUNE)
                  .batch(BATCH_SIZE).prefetch(AUTOTUNE))
            probs += model.predict(ds, verbose=0)
    return probs / (n_rot * 2)


def ensemble_weighted(preds_list, weights):
    w = np.abs(np.array(weights, dtype=float))
    w = w / (w.sum() + 1e-12)
    return sum(w[i] * preds_list[i] for i in range(len(preds_list)))


print("Fonctions définies.")

Fonctions définies.


## Cellule 4 — Charger les checkpoints (sautée sans GPU)

In [4]:
# Après la nouvelle phase 3 : remplace FINAL_PATH par le chemin du nouveau modèle
# (ex. "/kaggle/input/notebooks/rihembousbih/<copie>/effnetv2s_cbam_final.keras")
STEP1_PATH = f"{INPUT_DIR}/resnet50_cbam_finetune_step1.keras"
STEP2_PATH = f"{INPUT_DIR}/resnet50_cbam_finetune_step2.keras"
FINAL_PATH = f"{INPUT_DIR}/resnet50_cbam_final.keras"

if not HAS_GPU:
    print("⏭️  Pas de GPU → cellule sautée.")
else:
    val_df  = pd.read_csv(f"{INPUT_DIR}/split_val.csv")
    test_df = pd.read_csv(f"{INPUT_DIR}/split_test.csv")
    assert os.path.exists(test_df["path"].iloc[0]), \
        "❌ Images introuvables → ajoute en Input les datasets ISIC2019 / HAM10000 / PH2"
    load = lambda p: tf.keras.models.load_model(p, custom_objects=CUSTOM_OBJECTS, compile=False)
    MODELS      = [load(STEP1_PATH), load(STEP2_PATH), load(FINAL_PATH)]
    MODEL_NAMES = ["step1", "step2", "final"]
    print("3 modèles chargés.")

⏭️  Pas de GPU → cellule sautée.


## Cellule 5 — Ensemble pondéré (sautée sans GPU, ~25 min avec GPU)

In [5]:
from scipy.optimize import differential_evolution

CACHE = "/kaggle/working"

def preds_cached(model, name, df, split):
    f = f"{CACHE}/preds_{split}_{name}.npy"
    if os.path.exists(f):
        print(f"  {split}/{name} : chargé depuis le cache")
        return np.load(f)
    t0 = time.time()
    p = predict_tta(model, df)
    np.save(f, p)
    print(f"  {split}/{name} : {time.time()-t0:.0f}s")
    return p

if not HAS_GPU:
    print("⏭️  Pas de GPU → cellule sautée.")
else:
    print("Prédictions VAL…")
    preds_val_list = [preds_cached(m, n, val_df, "val") for m, n in zip(MODELS, MODEL_NAMES)]
    y_val_true = val_df["label"].values
    for n, p in zip(MODEL_NAMES, preds_val_list):
        print(f"  VAL {n:6s} Acc={accuracy_score(y_val_true, p.argmax(1)):.4f}  "
              f"MacroF1={f1_score(y_val_true, p.argmax(1), average='macro'):.4f}")

    def neg_macro_f1(weights):
        probs = ensemble_weighted(preds_val_list, weights)
        return -f1_score(y_val_true, probs.argmax(1), average="macro")

    result = differential_evolution(neg_macro_f1, bounds=[(0, 1)] * len(MODELS),
                                    seed=SEED, maxiter=40, tol=1e-4, workers=1)
    W = np.abs(result.x) / np.abs(result.x).sum()
    np.save(f"{CACHE}/ensemble_weights.npy", W)
    print("Poids optimisés (VAL):", dict(zip(MODEL_NAMES, W.round(3))))

    print("\nPrédictions TEST…")
    preds_test_list = [preds_cached(m, n, test_df, "test") for m, n in zip(MODELS, MODEL_NAMES)]
    y_test_true = test_df["label"].values
    p_final = preds_test_list[-1].argmax(1)
    p_ens   = ensemble_weighted(preds_test_list, W).argmax(1)
    print(f"TEST final seul : Acc={accuracy_score(y_test_true, p_final):.4f}  "
          f"MacroF1={f1_score(y_test_true, p_final, average='macro'):.4f}")
    print(f"TEST ensemble   : Acc={accuracy_score(y_test_true, p_ens):.4f}  "
          f"MacroF1={f1_score(y_test_true, p_ens, average='macro'):.4f}")

⏭️  Pas de GPU → cellule sautée.
